# Propellant prices cleanup

In [1]:
from pathlib import Path
import re

import pandas as pd


ANALYSIS_DIR = Path.cwd()
RAW_FILE = ANALYSIS_DIR / "data" / "raw" / "propellant-prices" / "Prisdatabase.xlsx"
OUTPUT_PATH = ANALYSIS_DIR / "data" / "propellant-prices.csv"

START_YEAR = 2010
DATA_SOURCE = RAW_FILE.name
TIME_PATTERN = re.compile(r"^\d{4}M\d{2}$")

SHEETS = {
    "petrol_price":   {"sheet": "Blyfri 95",   "scale": 1.0},
    "diesel_price":   {"sheet": "Autodiesel",  "scale": 1.0},
    "electric_price": {"sheet": "Månedlig El", "scale": 1 / 1000},
}

In [2]:
def parse_price_sheet(path: Path, sheet: str, scale: float) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name=sheet, header=None, usecols="A:B", dtype={0: str})
    raw.columns = ["period", "value"]
    mask = raw["period"].astype(str).str.match(TIME_PATTERN)
    df = raw[mask].copy()
    df["value"] = pd.to_numeric(df["value"], errors="coerce") * scale
    df["year"] = df["period"].str.slice(0, 4).astype(int)
    df["month"] = df["period"].str.slice(5, 7).astype(int)
    return df[["year", "month", "value"]]

In [3]:
def build_propellant_prices() -> pd.DataFrame:
    series = {}
    for col, meta in SHEETS.items():
        s = parse_price_sheet(RAW_FILE, meta["sheet"], meta["scale"])
        series[col] = s.rename(columns={"value": col})

    merged = None
    for col, df in series.items():
        merged = df if merged is None else merged.merge(df, on=["year", "month"], how="outer")

    merged = merged[merged["year"] >= START_YEAR].copy()
    merged["data_source"] = DATA_SOURCE
    merged = merged.sort_values(["year", "month"], kind="stable").reset_index(drop=True)
    return merged[["year", "month", "electric_price", "diesel_price", "petrol_price", "data_source"]]


propellant_prices = build_propellant_prices()
propellant_prices.head()

,year,month,electric_price,diesel_price,petrol_price,data_source
0,2010,1,0.397316,9.12,10.59,Prisdatabase.xlsx
1,2010,2,0.512986,9.16,10.65,Prisdatabase.xlsx
2,2010,3,0.423136,9.50,11.06,Prisdatabase.xlsx
3,2010,4,0.351907,9.81,11.25,Prisdatabase.xlsx
4,2010,5,0.319804,9.77,11.12,Prisdatabase.xlsx


In [4]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
propellant_prices.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH

PosixPath('/Users/marcuslauridsen/Developer/Github/social-data-final-project/analysis/data/propellant-prices.csv')

In [5]:
quality_report = {
    "rows": len(propellant_prices),
    "year_min": int(propellant_prices["year"].min()),
    "year_max": int(propellant_prices["year"].max()),
    "month_min": int(propellant_prices["month"].min()),
    "month_max": int(propellant_prices["month"].max()),
}

null_report = propellant_prices.isna().sum().to_dict()
quality_report, null_report

({'rows': 192,
  'year_min': 2010,
  'year_max': 2025,
  'month_min': 1,
  'month_max': 12},
 {'year': 0,
  'month': 0,
  'electric_price': 0,
  'diesel_price': 0,
  'petrol_price': 0,
  'data_source': 0})